# Step 3 – Sobol global sensitivity analysis

Inputs varied: T (log10 scale), R, I, IC and lactating herd size HS. Herd size uses herds built at each value of `HS_GRID`; a sampled HS uses the nearest one. Each parameter set is simulated `SOBOL_REPS` times for `SA_HORIZON_DAYS` and the outcomes are averaged.

**Outputs** (`results/sensitivity/`):

| File | Content |
| --- | --- |
| `sobol_samples_outputs.csv` | One row per parameter set: inputs and averaged outcomes |
| `sobol_indices.csv` | S1, ST and bootstrap 95% CIs per outcome and parameter, at each sample size in `SOBOL_CONVERGENCE_N` (convergence check) |
| `sobol_second_order.csv` | S2 and CIs per outcome and parameter pair, same sample sizes |

In [1]:
#!pip install SALib

In [2]:
import time
import warnings

import numpy as np
import pandas as pd

import ab_sim as m
import common as c
import config as cfg

from SALib.analyze import sobol as sobol_analyze
from SALib.sample import sobol as sobol_sample

from workers import SOBOL_NAMES, sobol_task

## Functions

In [3]:
def sobol_problem() -> dict:
    """SALib problem: T on log10 scale; integer parameters widened by 0.5 so each integer
    has equal weight after rounding."""
    bounds = []
    for p in SOBOL_NAMES:
        lo, hi = cfg.SOBOL_BOUNDS[p]
        if p in cfg.LOG_SCALE_PARAMS:
            lo, hi = np.log10(lo), np.log10(hi)
        elif p in cfg.INTEGER_PARAMS:
            lo, hi = lo - 0.5, hi + 0.5
        bounds.append([float(lo), float(hi)])
    return {"num_vars": len(SOBOL_NAMES), "names": SOBOL_NAMES, "bounds": bounds}

In [4]:
def sobol_indices(df: pd.DataFrame, prob: dict) -> tuple:
    """First-, total- and second-order indices with bootstrap 95% CIs, computed on the first
    n base samples for each n in SOBOL_CONVERGENCE_N (valid because SALib orders samples
    in blocks of 2k + 2 rows per base sample)."""
    block = 2 * prob["num_vars"] + 2
    first, second = [], []
    for n in sorted(set(cfg.SOBOL_CONVERGENCE_N) | {cfg.SOBOL_N}):
        if n > cfg.SOBOL_N:
            continue
        sub = df.iloc[: n * block]
        for outcome in c.OUTCOMES:
            Y = sub[outcome].to_numpy(float)
            if np.allclose(Y, Y[0]):
                continue
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                Si = sobol_analyze.analyze(prob, Y, calc_second_order=True, num_resamples=1000,
                                           conf_level=0.95, seed=cfg.SOBOL_SEED)
            for i, p in enumerate(SOBOL_NAMES):
                first.append({"N": n, "outcome": outcome, "parameter": p,
                              "S1": Si["S1"][i], "S1_conf95": Si["S1_conf"][i],
                              "ST": Si["ST"][i], "ST_conf95": Si["ST_conf"][i]})
                for j in range(i + 1, len(SOBOL_NAMES)):
                    second.append({"N": n, "outcome": outcome, "parameter_1": p,
                                   "parameter_2": SOBOL_NAMES[j], "S2": Si["S2"][i, j],
                                   "S2_conf95": Si["S2_conf"][i, j]})
    return pd.DataFrame(first), pd.DataFrame(second)

## Herds

In [5]:
c.ensure_dirs()
OUT = c.DIRS["sensitivity"]
c.farm_burn_ins(m.load_inputs(cfg), verbose=False)
grid = c.grid_burn_ins()
c.write_burnin_summary(grid)
pd.read_csv(c.DIRS["burnin"] / "burnin_summary_grid.csv")

Burn-in grid herd 500
Burn-in grid herd 1000
Burn-in grid herd 1500
Burn-in grid herd 2000
Burn-in grid herd 2500
Burn-in grid herd 3000


,herd,target_lactating,lactating_at_end,total_at_end,young_stock_target,pen1_Pre-weaning,pen2_Post-weaned,pen3_Breeding,pen4_Pregnant,pen5_Springer,pen6_Fresh primiparous,pen7_Fresh multiparous,pen8_High-milk primiparous,pen9_Low-milk primiparous,pen10_High-milk multiparous,pen11_Low-milk multiparous,pen12_Dry,pen13_Close-up,pen14_Calving,pen15_Hospital (post-calving)
0,500,500,500,1152,652,73,364,54,147,14,11,58,77,45,82,148,63,14,1,1
1,1000,1000,1000,2303,1303,147,727,108,294,27,17,89,132,87,213,327,109,15,6,5
2,1500,1500,1500,3455,1955,220,1091,162,441,41,30,134,214,125,292,506,150,25,11,13
3,2000,2000,2000,4608,2607,293,1455,216,589,55,42,170,323,187,392,642,191,34,7,12
4,2500,2500,2500,5759,3258,367,1818,270,736,68,45,230,407,210,481,798,246,42,21,20
5,3000,3000,3000,6910,3910,440,2182,323,883,82,52,279,486,255,593,968,286,50,16,15


## Sample and simulate

In [6]:
prob = sobol_problem()
X = sobol_sample.sample(prob, cfg.SOBOL_N, calc_second_order=True, seed=cfg.SOBOL_SEED)
print(f"Sobol: {len(X)} parameter sets x {cfg.SOBOL_REPS} simulations")

t0 = time.time()
rows = c.pool_map(sobol_task, list(enumerate(X)), include_grid=True, chunksize=8)
print(f"done in {time.time() - t0:.0f} s")

df = pd.DataFrame(rows).sort_values("row")
df.to_csv(OUT / "sobol_samples_outputs.csv", index=False)

Sobol: 12288 parameter sets x 3 simulations
done in 161 s


## Indices

In [7]:
first, second = sobol_indices(df, prob)
first.to_csv(OUT / "sobol_indices.csv", index=False)
second.to_csv(OUT / "sobol_second_order.csv", index=False)
first[first.N == cfg.SOBOL_N]

,N,outcome,parameter,S1,S1_conf95,ST,ST_conf95
90,1024,peak_daily_cases,T,0.262545,0.051903,0.392004,0.056254
91,1024,peak_daily_cases,R,0.003241,0.011639,0.017393,0.003939
92,1024,peak_daily_cases,I,0.065684,0.038043,0.158568,0.026050
93,1024,peak_daily_cases,IC,0.024338,0.017538,0.037305,0.006672
94,1024,peak_daily_cases,HS,0.485318,0.071999,0.636081,0.077918
95,1024,cumulative_cases,T,0.065894,0.027592,0.103095,0.012722
96,1024,cumulative_cases,R,0.005614,0.016516,0.037369,0.006644
97,1024,cumulative_cases,I,0.006951,0.015992,0.033038,0.005452
98,1024,cumulative_cases,IC,0.042822,0.024078,0.077689,0.011448
99,1024,cumulative_cases,HS,0.830856,0.074850,0.893385,0.061407
